# 🎯 Mock Interview Copilot - Backend API

This notebook deploys the FastAPI backend with AI models on Kaggle.

## Requirements
- **GPU**: Enable GPU T4 x2 in Settings → Accelerator
- **ngrok token**: Get from https://dashboard.ngrok.com

## How to Use
1. Run Cell 1 to install dependencies
2. Run Cell 2 to set your tokens
3. Run Cell 3 to start the API server
4. Copy the public URL to use in your frontend

In [ ]:
# Cell 1: Install Dependencies
!pip install -q fastapi uvicorn pyngrok python-multipart
!pip install -q transformers==4.44.0 accelerate bitsandbytes
!pip install -q sentence-transformers faiss-cpu PyPDF2
!pip install -q pydantic sentencepiece

print("✅ All dependencies installed!")

In [ ]:
# Cell 2: Configure Your Tokens
import os

# ⚠️ REQUIRED: Replace these with your actual values!
os.environ["NGROK_TOKEN"] = "YOUR_NGROK_TOKEN_HERE"  # Get from https://dashboard.ngrok.com
os.environ["API_KEY"] = "secret123"                   # Choose any secret key

print("✅ Tokens configured!")
print(f"   API Key: {os.environ['API_KEY']}")

In [ ]:
# Cell 3: Start Backend API Server
# This cell loads models and starts the server - keep it running!

import json
import os
import re
import socket
import tempfile
import threading
import time
from typing import Dict, List

import faiss
import numpy as np
import torch
import uvicorn
from fastapi import FastAPI, File, Form, HTTPException, Request, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse
from pyngrok import conf, ngrok
from PyPDF2 import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# =============================================================================
# CONFIGURATION
# =============================================================================
API_KEY = os.environ.get("API_KEY", "secret123")
NGROK_TOKEN = os.environ.get("NGROK_TOKEN")
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

# Global model references
model = None
tokenizer = None
embedder = None

# =============================================================================
# FASTAPI APP
# =============================================================================
app = FastAPI(title="Mock Interview Copilot API")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# =============================================================================
# UTILITY FUNCTIONS
# =============================================================================

def extract_pdf_text(pdf_bytes: bytes) -> str:
    """Extract text from PDF bytes."""
    with tempfile.NamedTemporaryFile(delete=False, suffix=".pdf") as tmp:
        tmp.write(pdf_bytes)
        tmp_path = tmp.name
    try:
        reader = PdfReader(tmp_path)
        text = "\n".join(page.extract_text() or "" for page in reader.pages)
    finally:
        os.remove(tmp_path)
    return text.strip()


def chunk_text(text: str, chunk_size: int = 200, overlap: int = 40) -> List[str]:
    """Split text into overlapping chunks."""
    words = text.split()
    chunks = []
    step = max(1, chunk_size - overlap)
    for i in range(0, len(words), step):
        chunk = " ".join(words[i : i + chunk_size])
        if chunk.strip():
            chunks.append(chunk)
    return chunks


def compute_relevance(question: str, index, chunks: List[str], k: int = 3) -> float:
    """Compute relevance score using FAISS."""
    query_emb = embedder.encode([question], convert_to_numpy=True)
    distances, _ = index.search(query_emb, k)
    avg_dist = float(distances.mean())
    return min(max(1 / (1 + avg_dist), 0.0), 1.0)


def build_prompt(resume: str, job_desc: str) -> str:
    """Build the LLM prompt for question generation."""
    return f"""<|im_start|>system
You are an expert technical interviewer. Generate interview questions that assess BOTH the candidate's experience AND the job requirements.

- (IMPORTANT) GENERATE 5 Technical questions and 3 behavioral questions

CRITICAL RULES:
1. For RESUME-BASED questions:
   - Use SPECIFIC technologies, projects, tools from the resume
   - Ask about their ACTUAL experience and results
   - Reference real details (no placeholders)

2. For JOB-BASED questions:
   - Focus on skills/technologies mentioned in job description
   - Ask how they would approach problems relevant to the role
   - Test knowledge needed for the position

3. For ALL suggested_answers:
   - Write COMPLETE, DETAILED answers (150-200 words)
   - Use first person ("I", "We", "My team")
   - Include specific technical details
   - Show problem-solving process
   - Demonstrate measurable results

4. Output ONLY valid JSON

<|im_end|>
<|im_start|>user
CANDIDATE RESUME:
{resume[:1500]}

TARGET JOB DESCRIPTION:
{job_desc}

Generate 5 technical questions:
- 2 questions: Based on candidate's resume experience (specific projects/tech they've used)
- 3 questions: Based on job requirements (skills needed for role, theoretical/practical)

Generate 3 behavioral questions:
- 1 question: About specific experience from their resume
- 1 question: About teamwork/collaboration
- 1 question: About handling challenges or learning new skills

For EACH question, provide:
- question: Clear, specific question
- suggested_answer: Complete example answer (150-200 words) showing how candidate could respond
- relevance_score: 0.0-1.0 (higher = more relevant to this specific role)

JSON FORMAT:
{{
  "technical_questions": [
    {{
      "question": "Tell me about your experience with [specific tech from resume] in your [specific project from resume].",
      "suggested_answer": "In my role at [Company from resume], I worked extensively with [technology]...",
      "relevance_score": 0.85,
      "question_type": "resume_based"
    }}
  ],
  "behavioral_questions": [
    {{
      "question": "Describe a challenging technical problem you solved in your [specific project from resume].",
      "suggested_answer": "In [specific project from resume], we faced [specific challenge]...",
      "relevance_score": 0.80,
      "question_type": "resume_based"
    }}
  ]
}}

IMPORTANT:
- Generate EXACTLY 5 technical questions and 3 behavioral questions
- Mix resume-based and job-based questions
- Use REAL details from resume and ACTUAL requirements from job description
- NO placeholders like [specific tech from resume] in the actual output

JSON:
<|im_end|>
<|im_start|>assistant
"""


# =============================================================================
# PARSING FUNCTIONS (from original working code)
# =============================================================================

def extract_skills_from_text(text: str) -> List[str]:
    """Extract skills from resume text."""
    skills = []
    skill_section_patterns = [
        r"(?:skills|technologies|technical skills|expertise)[\s:]*([^\n]+(?:\n[^\n]+){0,10})",
        r"(?:proficient in|experienced with|familiar with)[\s:]*([^\n]+)",
    ]
    
    for pattern in skill_section_patterns:
        matches = re.findall(pattern, text, re.IGNORECASE)
        for match in matches:
            potential = re.split(r"[,;|•\n]", match)
            for skill in potential:
                skill = re.sub(r"^[-•*\s]+", "", skill.strip())
                if 3 <= len(skill) <= 50 and not skill.isdigit():
                    skills.append(skill)
    
    # Extract technology names (CamelCase, etc.)
    tech_pattern = r"\b[A-Z][a-zA-Z]*(?:\.[A-Z][a-zA-Z]*)*\b|\b[A-Z][a-z]+(?:[A-Z][a-z]+)+\b"
    tech_matches = re.findall(tech_pattern, text)
    common_words = {"Bachelor", "Master", "University", "College", "Project", "Work",
                    "Experience", "Education", "Skills", "Summary", "Objective"}
    
    for tech in tech_matches:
        if tech not in common_words and len(tech) >= 3:
            skills.append(tech)
    
    # Remove duplicates
    unique = []
    seen = set()
    for skill in skills:
        lowered = skill.lower()
        if lowered not in seen:
            seen.add(lowered)
            unique.append(skill)
    return unique[:15]


def extract_projects_from_text(text: str) -> List[str]:
    """Extract projects from resume text."""
    projects = []
    project_patterns = [
        r"(?:project|projects)[\s:]*\n([^\n]+(?:\n(?!\n)[^\n]+)*)",
        r"(?:developed|built|created|implemented)\s+([^\.\n]{10,80})",
    ]
    for pattern in project_patterns:
        matches = re.findall(pattern, text, re.IGNORECASE)
        for match in matches:
            cleaned = match.strip()[:100]
            if cleaned and len(cleaned) > 10:
                projects.append(cleaned)
    return projects[:5]


def extract_companies_from_text(text: str) -> List[str]:
    """Extract companies/job titles from resume text."""
    companies = []
    lines = text.split("\n")
    job_keywords = ["intern", "developer", "engineer", "analyst", "specialist", 
                    "manager", "consultant", "designer", "architect", "lead"]
    for idx, line in enumerate(lines):
        lower = line.lower()
        if any(keyword in lower for keyword in job_keywords):
            company_line = line.strip()
            if idx + 1 < len(lines):
                next_line = lines[idx + 1].strip()
                if next_line and len(next_line) < 60 and not next_line[0].isdigit():
                    company_line = f"{company_line} at {next_line}"
            if len(company_line) > 5:
                companies.append(company_line[:80])
    return companies[:3]


def parse_llm_response(raw_output: str, resume_text: str, job_description: str) -> Dict:
    """Parse JSON from LLM response with robust fallback."""
    # Find best starting position using multiple markers
    markers = [
        "<|im_start|>assistant",
        "JSON:",
        "assistant\n",
        "<|im_end|>",
    ]
    start_pos = 0
    for marker in markers:
        pos = raw_output.rfind(marker)
        if pos != -1:
            start_pos = max(start_pos, pos + len(marker))
    
    text_to_parse = raw_output[start_pos:].strip()
    
    # Clean up markdown code blocks
    text_to_parse = re.sub(r"```json\s*", "", text_to_parse)
    text_to_parse = re.sub(r"```\s*", "", text_to_parse)
    text_to_parse = text_to_parse.replace("<|im_end|>", "")
    
    # Find JSON boundaries
    start_idx = text_to_parse.find("{")
    end_idx = text_to_parse.rfind("}")
    
    if start_idx == -1 or end_idx == -1 or start_idx >= end_idx:
        print("⚠️ No valid JSON found, using smart fallback")
        return create_smart_fallback(resume_text, job_description)
    
    json_str = text_to_parse[start_idx : end_idx + 1]
    
    try:
        parsed = json.loads(json_str)
    except json.JSONDecodeError as e:
        print(f"⚠️ JSON decode error: {e}")
        # Try to fix common JSON issues
        try:
            # Fix trailing commas
            fixed = re.sub(r',(\s*[}\]])', r'\1', json_str)
            parsed = json.loads(fixed)
        except:
            print("⚠️ JSON fix failed, using smart fallback")
            return create_smart_fallback(resume_text, job_description)
    
    if not isinstance(parsed, dict):
        return create_smart_fallback(resume_text, job_description)
    
    # Ensure required keys exist
    for key in ["technical_questions", "behavioral_questions"]:
        parsed.setdefault(key, [])
    
    # Check for placeholder questions (LLM didn't fill in properly)
    def has_placeholders(items):
        for item in items:
            question = item.get("question", "") if isinstance(item, dict) else ""
            if "[" in question or "from resume" in question.lower():
                return True
        return False
    
    if has_placeholders(parsed["technical_questions"]) or has_placeholders(parsed["behavioral_questions"]):
        print("⚠️ LLM left placeholders, using smart fallback")
        return create_smart_fallback(resume_text, job_description)
    
    # Validate we have enough questions
    if len(parsed["technical_questions"]) < 3 or len(parsed["behavioral_questions"]) < 2:
        print("⚠️ Not enough questions generated, using smart fallback")
        return create_smart_fallback(resume_text, job_description)
    
    return parsed


def create_smart_fallback(resume_text: str, job_description: str) -> Dict:
    """Create personalized fallback questions based on resume content."""
    skills = extract_skills_from_text(resume_text)
    projects = extract_projects_from_text(resume_text)
    companies = extract_companies_from_text(resume_text)
    
    primary_skill = skills[0] if skills else "your technical skills"
    secondary_skills = ", ".join(skills[1:3]) if len(skills) > 1 else "various technologies"
    tertiary_skill = skills[3] if len(skills) > 3 else "relevant technologies"
    project_context = projects[0] if projects else "your projects"
    company_context = companies[0] if companies else "your work experience"
    job_preview = job_description[:100] if job_description else "the target role"

    return {
        "technical_questions": [
            {
                "question": f"Can you walk me through your experience with {primary_skill}? What specific challenges did you face and how did you overcome them?",
                "suggested_answer": (
                    f"In my work with {primary_skill}, I encountered a significant challenge when building {project_context}. "
                    "The main issue was optimizing performance while maintaining readability. I profiled the application, tuned queries, and "
                    "implemented caching, which reduced response times by over 3x while keeping the codebase maintainable."
                ),
                "relevance_score": 0.82,
            },
            {
                "question": f"I see you worked on {project_context}. How would you apply that experience to this role, especially regarding {secondary_skills}?",
                "suggested_answer": (
                    f"During {project_context}, I owned the backend services that relied heavily on {secondary_skills}. "
                    "I designed event-driven services, introduced monitoring, and optimized latency. Those lessons transfer directly to your stack, "
                    "particularly around scaling and instrumentation."
                ),
                "relevance_score": 0.78,
            },
            {
                "question": f"How would you approach designing a scalable system for {job_preview}? What technologies would you choose and why?",
                "suggested_answer": (
                    "I start by mapping workloads, then partition the architecture into independently scalable services. "
                    f"For {job_preview}, I would pair a reliable relational store with a caching tier, asynchronous workers, and granular observability "
                    "so we can iterate based on real signals."
                ),
                "relevance_score": 0.85,
            },
            {
                "question": f"What's your experience with {tertiary_skill}? Can you describe a situation where you had to optimize code or system performance?",
                "suggested_answer": (
                    f"I used {tertiary_skill} extensively to diagnose a latency spike. Profiling revealed inefficient database access, so I reworked indexes, "
                    "added batching, and introduced connection pooling, cutting P95 latency from seconds to milliseconds."
                ),
                "relevance_score": 0.8,
            },
            {
                "question": "How do you ensure code quality and maintainability in your projects? What testing strategies do you follow?",
                "suggested_answer": (
                    "I enforce automated checks (formatting, linting, unit + integration tests) and keep documentation close to the code. "
                    "CI gates every change, and we review architectural decisions so the team shares context."
                ),
                "relevance_score": 0.83,
            },
        ],
        "behavioral_questions": [
            {
                "question": f"Tell me about your experience at {company_context}. Describe a situation where you had to collaborate with others to solve a complex problem.",
                "suggested_answer": (
                    f"At {company_context}, an intermittent production incident affected key clients. I coordinated backend, DevOps, and QA teammates, "
                    "set up a war room, traced the race condition in our cache, and shipped a fix with targeted regression tests."
                ),
                "relevance_score": 0.8,
            },
            {
                "question": "Tell me about a time when you had to work with a difficult team member or stakeholder. How did you handle the situation?",
                "suggested_answer": (
                    "I schedule a dedicated conversation, surface data behind each option, and highlight common goals. "
                    "By acknowledging valid concerns and offering compromises where possible, we keep discussions constructive."
                ),
                "relevance_score": 0.78,
            },
            {
                "question": f"Describe a time when you had to quickly learn a new technology or adapt to significant changes while working on {project_context}.",
                "suggested_answer": (
                    "I built a structured learning plan, paired with experienced teammates, and documented every insight in our wiki. "
                    "Within weeks I became the go-to contact for that component, proving the value of deliberate practice and knowledge sharing."
                ),
                "relevance_score": 0.77,
            },
        ],
    }


# =============================================================================
# API ENDPOINTS
# =============================================================================

@app.get("/")
async def root():
    """Health check endpoint."""
    return {
        "status": "online",
        "message": "Mock Interview Copilot API",
        "model_loaded": model is not None
    }


@app.post("/interview")
async def generate_interview(
    request: Request,
    file: UploadFile = File(...),
    job_description: str = Form(...)
):
    """Generate personalized interview questions."""
    # Authenticate
    auth = request.headers.get("authorization", "")
    if not auth.startswith("Bearer ") or auth[7:] != API_KEY:
        raise HTTPException(status_code=401, detail="Invalid API key")
    
    # Validate
    if not model:
        raise HTTPException(status_code=503, detail="Models still loading, please wait")
    if not file.filename.lower().endswith(".pdf"):
        raise HTTPException(status_code=400, detail="Only PDF files are supported")
    if len(job_description.strip()) < 20:
        raise HTTPException(status_code=400, detail="Job description too short (min 20 chars)")
    
    # Extract resume text
    pdf_bytes = await file.read()
    resume_text = extract_pdf_text(pdf_bytes)
    if not resume_text:
        raise HTTPException(status_code=400, detail="Could not extract text from PDF")
    
    # Create embeddings and FAISS index
    chunks = chunk_text(resume_text)
    embeddings = embedder.encode(chunks, convert_to_numpy=True)
    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(embeddings)
    
    # Generate with LLM
    prompt = build_prompt(resume_text, job_description)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    print("🤖 Generating questions with LLM...")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=2100,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    raw_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"📝 Raw output length: {len(raw_output)} chars")
    
    # Parse response with smart fallback
    parsed = parse_llm_response(raw_output, resume_text, job_description)
    
    # Compute/update relevance scores
    for bucket in ["technical_questions", "behavioral_questions"]:
        for q in parsed.get(bucket, []):
            if not q.get("relevance_score"):
                q["relevance_score"] = compute_relevance(
                    q.get("question", ""), index, chunks
                )
            q["relevance_score"] = round(float(q["relevance_score"]), 3)
    
    print("✅ Questions generated successfully")
    return JSONResponse(content=parsed)


# =============================================================================
# MODEL LOADING
# =============================================================================

def load_models():
    """Load AI models."""
    global model, tokenizer, embedder
    
    print("🔎 Loading embedding model...")
    embedder = SentenceTransformer(EMBED_MODEL)
    print("✅ Embedding model loaded")
    
    print(f"🚀 Loading LLM: {MODEL_ID}...")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.bfloat16,
    ).eval()
    
    print("✅ LLM loaded successfully")


def get_free_port():
    """Find an available port."""
    s = socket.socket()
    s.bind(("", 0))
    port = s.getsockname()[1]
    s.close()
    return port


# =============================================================================
# STARTUP
# =============================================================================

# Load models
print("="*60)
print("🎯 MOCK INTERVIEW COPILOT - STARTING")
print("="*60)

load_models()

# Start ngrok tunnel
port = get_free_port()
conf.get_default().auth_token = NGROK_TOKEN
public_url = ngrok.connect(port).public_url

print("\n" + "="*60)
print("✅ API SERVER IS RUNNING!")
print("="*60)
print(f"\n🌐 Public URL: {public_url}")
print(f"📋 API Docs:   {public_url}/docs")
print(f"🔑 API Key:    {API_KEY}")
print("\n📋 Copy these values to your frontend:")
print("-"*60)
print(f'API_URL = "{public_url}/interview"')
print(f'API_KEY = "{API_KEY}"')
print("-"*60)
print("\n⚠️  Keep this cell running to maintain the connection!")
print("="*60)

# Run server in background thread
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=port, log_level="warning")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

# Keep alive
try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\n🛑 Shutting down...")
    ngrok.kill()